# Isharah-500 canonical SI dev evaluation

This notebook explains the canonical token-level CTC evaluation and visualizes the saved corrected SI dev report. It uses the SI dev split only. It never loads SI test metadata or test labels.

## Metrics

For each reference/hypothesis pair, the canonical implementation aligns gloss tokens with deterministic Levenshtein backtracking. Substitutions, deletions, and insertions are counted from edit operations.

- GER = `(substitutions + deletions + insertions) / total reference glosses`
- Exact Match = complete sequence matches / total samples
- Repeated glosses are handled by sequence equality and CTC decoding rules, not naive positional comparison.

In [ ]:
from pathlib import Path
import json
import os
def resolve_project_root():
    configured = os.environ.get("ISHARAH_PROJECT_ROOT")
    candidates = [Path(configured)] if configured else []
    current = Path.cwd().resolve()
    candidates.extend([current, *current.parents])
    for candidate in candidates:
        if (candidate / "isharah500" / "src").is_dir():
            return candidate
    raise RuntimeError("Set ISHARAH_PROJECT_ROOT to the repository worktree root.")

PROJECT_ROOT = resolve_project_root()
ISHARAH_ROOT = PROJECT_ROOT / "isharah500"
REPORT_PATH = ISHARAH_ROOT / "reports" / "si_error_analysis_epoch15_corrected.json"
report = json.loads(REPORT_PATH.read_text(encoding="utf-8"))
required_report = {"protocol", "split", "sample_count", "analysis"}
required_aggregate = {"substitutions", "deletions", "insertions", "gloss_error_rate", "exact_match"}
if not required_report.issubset(report) or not required_aggregate.issubset(report["analysis"]["aggregate"]):
    raise ValueError(f"Unexpected corrected report schema: {REPORT_PATH}")
assert report["protocol"] == "SI"
assert report["split"] == "dev"
assert report["sample_count"] == 500
aggregate = report["analysis"]["aggregate"]
print(aggregate)

In [ ]:
import matplotlib.pyplot as plt
operations = ["substitutions", "deletions", "insertions"]
counts = [aggregate[key] for key in operations]
plt.figure(figsize=(7, 4))
plt.bar(operations, counts)
plt.title("Corrected SI dev edit operations")
plt.ylabel("count across 500 dev samples")
plt.show()

## Sentence-length failure patterns

The saved report groups metrics by target gloss length. Longer sequences show increasing deletion rates and lower exact-match performance. The next cells visualize those saved groups without reopening any dataset split.

In [ ]:
by_length = report["analysis"]["by_target_length"]
lengths = sorted(by_length, key=int)
deletions = [by_length[length]["deletion_rate"] for length in lengths]
exact_matches = [by_length[length]["exact_match"] for length in lengths]
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot([int(x) for x in lengths], deletions, marker="o")
axes[0].set_title("Deletion rate by target length")
axes[0].set_xlabel("target gloss length")
axes[0].set_ylabel("deletion rate")
axes[1].plot([int(x) for x in lengths], exact_matches, marker="o")
axes[1].set_title("Exact Match by target length")
axes[1].set_xlabel("target gloss length")
axes[1].set_ylabel("exact match")
for axis in axes:
    axis.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Documented inference API demonstration

This optional cell loads the selected checkpoint once and predicts one real SI dev keypoint sequence. It does not compare against a label, calculate a test metric, or access SI test data. Configure `ISHARAH_DATA_ROOT` before execution.

In [ ]:
# Optional real-data inference cell; no execution is performed automatically.
import csv
import sys
import numpy as np
data_root_value = os.environ.get("ISHARAH_DATA_ROOT")
if not data_root_value:
    raise RuntimeError("Set ISHARAH_DATA_ROOT before running this optional cell.")
data_root = Path(data_root_value).resolve()
source_root = ISHARAH_ROOT / "src"
if str(source_root) not in sys.path:
    sys.path.insert(0, str(source_root))
from isharah_inference import load_predictor
with (ISHARAH_ROOT / "metadata" / "isharah500_si_dev.csv").open(encoding="utf-8-sig", newline="") as handle:
    dev_row = next(csv.DictReader(handle))
relative = Path(dev_row["keypoints_path"])
keypoint_path = data_root / relative
if not keypoint_path.is_file():
    raise FileNotFoundError(keypoint_path)
checkpoint = ISHARAH_ROOT / "checkpoints" / "si_hand_centric_resume_15ep" / "isharah500_si_best.pt"
predictor = load_predictor(checkpoint, device_name="cuda")
prediction = predictor.predict(np.load(keypoint_path, allow_pickle=False))
prediction.to_dict()

The expected output schema is `gloss_tokens`, `gloss_text`, `input_length`, `valid_hand_coverage`, `device`, and `preprocessing_config`. The notebook does not claim a confidence score or an accuracy result from this single demonstration.